In [33]:
import os
import re
import pathlib
import pandas as pd
from dotenv import dotenv_values
from sqlalchemy import text, create_engine

In [34]:
root_path = pathlib.Path(os.getcwd())
root_path = root_path.parents[1]

In [35]:
config = dotenv_values(root_path / ".env.local")

engine = create_engine(
    f"mysql+pymysql://{config['DB_USER']}:{config['DB_PASSWORD']}@{config['DB_HOST']}:{config['DB_PORT']}/{config['DB_NAME']}"
)

connection_database = engine.connect()

In [36]:
dataframe = pd.read_excel(
    os.path.join(
        root_path,
        "data_folder",
        "product_detail_export_cleaned.xlsx",
    )
)

# Table `Distributor`
> - `id_distributor` (primary key, auto-increment)
> - `distributor_name` (string, not null)

In [37]:
select_distributor = text("SELECT * FROM distributor")
database_distributor = pd.read_sql(select_distributor, connection_database)

In [38]:
dataframe_distributor = pd.Series(dataframe["distributor"].unique(), name="distributor_name")

In [39]:
insert_distributor = text(
    "INSERT INTO distributor (distributor_name) VALUES (:distributor_name)"
)

for distributor_name in dataframe_distributor:
    connection_database.execute(insert_distributor, {"distributor_name": distributor_name})

connection_database.commit()

# TABLE `Industrial`

> - `id_industrial` (primary key, auto-increment)
> - `industrial_name` (string, not null)

In [40]:
select_industrial = text("SELECT * FROM industrial")
database_industrial = pd.read_sql(select_industrial, connection_database)

In [41]:
dataframe_industrial = pd.Series(dataframe["industrial"].unique(), name="industrial_name")

In [42]:
insert_industrial = text(
    "INSERT INTO industrial (industrial_name) VALUES (:industrial_name)"
)

for industrial_name in dataframe_industrial:
    connection_database.execute(insert_industrial, {"industrial_name": industrial_name})

connection_database.commit()

# Table `Brand`

> - `id_brand` (primary key, auto-increment)
> - `brand_name` (string, not null)

In [43]:
select_brand = text("SELECT * FROM brand")
database_brand = pd.read_sql(select_brand, connection_database)

In [44]:
dataframe_brand = pd.Series(dataframe["brand"].unique(), name="brand_name")

In [45]:
insert_brand = text(
    "INSERT INTO brand (brand_name) VALUES (:brand_name)"
)

for brand_name in dataframe_brand:
    connection_database.execute(insert_brand, {"brand_name": brand_name})

connection_database.commit()

# Table `Unit`

> - `id_unit` (primary key, auto-increment)
> - `unit_name` (string, not null)

In [46]:
select_unit = text("SELECT * FROM unit")
database_unit = pd.read_sql(select_unit, connection_database)

In [47]:
dataframe_unit = pd.Series(dataframe["unit"].unique(), name="unit_name")

In [48]:
insert_unit = text(
    "INSERT INTO unit (unit_name) VALUES (:unit_name)"
)

for unit_name in dataframe_unit:
    connection_database.execute(insert_unit, {"unit_name": unit_name})

connection_database.commit()

# Table `Data Source`

> - `id_data_source` (primary key, auto-increment)
> - `data_source_name` (string, not null)

In [49]:
select_data_source = text("SELECT * FROM data_source")
database_data_source = pd.read_sql(select_data_source, connection_database)

In [50]:
dataframe_data_source = pd.Series(dataframe["data_source"].unique(), name="data_source_name")

In [51]:
insert_data_source = text(
    "INSERT INTO data_source (data_source_name) VALUES (:data_source_name)"
)

for data_source_name in dataframe_data_source:
    connection_database.execute(insert_data_source, {"data_source_name": data_source_name})

connection_database.commit()

# Table `Category`

> - `id_category` (primary key, auto-increment)
> - `category_name` (string, not null)

In [52]:
select_category = text("SELECT * FROM category")
database_category = pd.read_sql(select_category, connection_database)

In [53]:
mapping = pd.read_excel(
    os.path.join(root_path, "data_folder", "mapping", "mapping_product.xlsx"),
    sheet_name="mapping_products",
)

dataframe_category = pd.Series(mapping["categories"].unique(), name="category_name")

In [54]:
insert_category = text(
    "INSERT INTO category (category_name) VALUES (:category_name)"
)

for category_name in dataframe_category:
    connection_database.execute(insert_category, {"category_name": category_name})

connection_database.commit()

# Table `Product`

> - `id_product` (primary key, auto-increment)
> - `fk_id_brand` → `brand.id_brand`
> - `fk_id_category` → `category.id_category`
> - `fk_id_unit` → `unit.id_unit`
> - `fk_id_data_source` → `data_source.id_data_source`
> - `product_name` (string, nullable)
> - `product_code` (string, nullable)
> - `description` (text, nullable)

In [55]:
# Load final dataframe (product_name mapped) + reload reference tables with their DB ids
dataframe_final = pd.read_excel(
    os.path.join(root_path, "data_folder", "product_detail_export_final.xlsx")
)

database_brand       = pd.read_sql(text("SELECT * FROM brand"),       connection_database)
database_category    = pd.read_sql(text("SELECT * FROM category"),    connection_database)
database_unit        = pd.read_sql(text("SELECT * FROM unit"),        connection_database)
database_data_source = pd.read_sql(text("SELECT * FROM data_source"), connection_database)

# --- Verify if all brands in the final file exist in the database ---
brands_in_database  = set(database_brand["brand_name"])
brands_in_file = set(dataframe_final["brand"].dropna().unique())
missing_brands = brands_in_file - brands_in_database

if missing_brands:
    for brand in missing_brands:
        connection_database.execute(text("INSERT INTO brand (brand_name) VALUES (:name)"), {"name": brand})

    connection_database.commit()
    database_brand = pd.read_sql(text("SELECT * FROM brand"), connection_database)

# --- If any category in the final file is missing in DB, insert it (ex: "Non catégorisé") ---
if "Non catégorisé" not in set(database_category["category_name"]):
    connection_database.execute(
        text("INSERT INTO category (category_name) VALUES (:name)"),
        {"name": "Non catégorisé"},
    )
    connection_database.commit()
    database_category = pd.read_sql(text("SELECT * FROM category"), connection_database)

In [ ]:
dataframe_product = (
    dataframe_final
    [["product_name", "product_code", "description", "brand", "unit", "data_source"]]
    .copy() # copy for avoiding SettingWithCopyWarning in next steps (add new foreign keys)
)

# Step 1: join category from mapping on product_name (exact match)
dataframe_product = dataframe_product.merge(
    mapping[["product_name", "categories"]],
    on="product_name",
    how="left"
)

# Step 2: fallback — keyword matching on description for rows still without category
def find_category_by_keywords(param_description):
    """ Retourne la catégorie correspondante à une description de produit, en se basant sur les mots-clés définis dans le mapping """

    if pd.isna(param_description):
        return None

    description_upper = param_description.upper()
    for _, mapping_row in mapping.iterrows():
        brand_keyword = str(mapping_row["keywords_brands"]).upper()
        other_keywords = str(mapping_row["keywords_others"]).upper().split(";")

        if brand_keyword in description_upper and any(keyword.strip() in description_upper for keyword in other_keywords):
            return mapping_row["categories"]

    return None

mask_no_category = dataframe_product["categories"].isna()
dataframe_product.loc[mask_no_category, "categories"] = (
    dataframe_product.loc[mask_no_category, "description"]
    .apply(find_category_by_keywords)
)

# Step 3: When no category found after keyword matching, assign "Non catégorisé"
dataframe_product["categories"] = dataframe_product["categories"].fillna("Non catégorisé")

# Step 4: merge with database tables to get foreign keys (id_brand, id_category, id_unit, id_data_source)
dataframe_product = dataframe_product.merge(
    database_brand.rename(columns={"brand_name": "brand"}),
    on="brand",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_category.rename(columns={"category_name": "categories"}),
    on="categories",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_unit.rename(columns={"unit_name": "unit"}),
    on="unit",
    how="left"
)

dataframe_product = dataframe_product.merge(
    database_data_source.rename(columns={"data_source_name": "data_source"}),
    on="data_source",
    how="left"
)

# Step 5: extraire units_per_case depuis la description
# Pattern 1 : "10MLx200", "700g*12"  → \d+[lettres][xX*]\d+  (unité avant le séparateur)
# Pattern 2 : "Bte3/1X6"             → \d+[xX*]\d+           (chiffre directement avant le séparateur)
# Pattern 3 : "Elephant X50", "(20S Pyr)X12" → [xX]\d+       (xX précédé d'un caractère non-alphanumérique)
UPC_PATTERN = re.compile(r'\d+[A-Za-z]+[xX*](\d+)|\d+[xX*](\d+)|(?<![A-Za-z\d])[xX](\d+)')

def extract_units_per_case(param_description):
    """ Extracts the number of units per case from a product description string """

    if pd.isna(param_description):
        return 1

    unit_match = UPC_PATTERN.search(str(param_description))
    if unit_match:
        unit_value = unit_match.group(1) or unit_match.group(2) or unit_match.group(3)
        return int(unit_value)

    return 1

dataframe_product["units_per_case"] = dataframe_product["description"].apply(extract_units_per_case)

In [57]:
# Insert data into product table
insert_product = text("""
    INSERT INTO product
        (fk_id_brand, fk_id_category, fk_id_unit, fk_id_data_source,
        product_name, product_code, description, units_per_case)
    VALUES
        (:fk_id_brand, :fk_id_category, :fk_id_unit, :fk_id_data_source,
        :product_name, :product_code, :description, :units_per_case)
""")

for _, row in dataframe_product.iterrows():
    connection_database.execute(insert_product, {
        "fk_id_brand":       int(row["id_brand"]),
        "fk_id_category":    int(row["id_category"]),
        "fk_id_unit":        int(row["id_unit"]),
        "fk_id_data_source": int(row["id_data_source"]),
        "product_name":      row["product_name"]  if pd.notna(row["product_name"])  else None,
        "product_code":      row["product_code"]  if pd.notna(row["product_code"])  else None,
        "description":       row["description"]   if pd.notna(row["description"])   else None,
        "units_per_case":    int(row["units_per_case"]),
    })

connection_database.commit()

# Table `Agreement`

> - `id_agreement` (primary key, auto-increment)
> - `fk_id_brand` → `brand.id_brand`
> - `fk_id_category` → `category.id_category`
> - `fk_id_industrial` → `industrial.id_industrial`
> - `fk_id_unit` → `unit.id_unit`
> - `start_date` (date, nullable)
> - `end_date` (date, nullable)

In [58]:
database_brand      = pd.read_sql(text("SELECT * FROM brand"),      connection_database)
database_category   = pd.read_sql(text("SELECT * FROM category"),   connection_database)
database_industrial = pd.read_sql(text("SELECT * FROM industrial"), connection_database)
database_unit       = pd.read_sql(text("SELECT * FROM unit"),       connection_database)

mapping = pd.read_excel(
    os.path.join(root_path, "data_folder", "mapping", "mapping_product.xlsx"),
    sheet_name="mapping_products",
)

In [59]:
brand_name_correction = {"Hellmanns": "Hellmann's"}

brands_in_mapping  = {brand_name_correction.get(brand, brand) for brand in mapping["brand"].unique()}
brands_in_database = set(database_brand["brand_name"])
missing_brands     = brands_in_mapping - brands_in_database

for brand_name in missing_brands:
    connection_database.execute(
        text("INSERT INTO brand (brand_name) VALUES (:name)"),
        {"name": brand_name}
    )

if missing_brands:
    connection_database.commit()
    database_brand = pd.read_sql(text("SELECT * FROM brand"), connection_database)

if "UVC" not in set(database_unit["unit_name"]):
    connection_database.execute(text("INSERT INTO unit (unit_name) VALUES (:name)"), {"name": "UVC"})
    connection_database.commit()
    database_unit = pd.read_sql(text("SELECT * FROM unit"), connection_database)

In [ ]:
dataframe_final_ref = pd.read_excel(
    os.path.join(root_path, "data_folder", "product_detail_export_final.xlsx")
)

# Group by brand and take the first industrial for each brand to create a mapping dictionary
brand_to_industrial = dataframe_final_ref.groupby("brand")["industrial"].first().to_dict()

# Keep first industrial in DB
default_industrial  = database_industrial["industrial_name"].iloc[0]

# Get id_unit for UVC for inserting in agreement table
id_unit_uvc = int(database_unit[database_unit["unit_name"] == "UVC"]["id_unit"].iloc[0])

mapping_work = mapping.copy() # copy for add/modify columns without affecting original mapping dataframe

# Corrected brand names for mapping with DB (ex: "Hellmanns" in mapping vs "Hellmann's" in DB)
mapping_work["brand_db"]        = mapping_work["brand"].map(lambda brand: brand_name_correction.get(brand, brand))

# Map industrial names based on brand using the mapping dictionary
mapping_work["industrial_name"] = mapping_work["brand_db"].map(
    lambda brand: brand_to_industrial.get(brand, default_industrial)
)

# Merge for getting foreign keys (id_brand, id_category, id_industrial) for the agreement table
mapping_work = mapping_work.merge(
    database_brand.rename(columns={"brand_name": "brand_db"}),
    on="brand_db",
    how="left"
)
mapping_work = mapping_work.merge(
    database_category.rename(columns={"category_name": "categories"}),
    on="categories",
    how="left"
)
mapping_work = mapping_work.merge(
    database_industrial[["id_industrial", "industrial_name"]],
    on="industrial_name",
    how="left"
)

# Détecter le palier_group par ligne (préfixe de la première colonne palier non-NaN)
palier_columns_name = [column_name for column_name in mapping.columns if column_name.startswith("palier_")]

def _detect_palier_group(param_row):
    for palier_column_name in palier_columns_name:
        if pd.notna(param_row.get(cpalier_column_namel)):
            match_value = re.match(r"^palier_(.+)_(?:\d+-\d+|superior-\d+)_uvc$", palier_column_name)
            if match_value:
                return match_value.group(1)
    return None

mapping_work["palier_group"] = mapping_work.apply(_detect_palier_group, axis=1)

insert_agreement = text("""
    INSERT INTO agreement
        (fk_id_brand, fk_id_category, fk_id_industrial, fk_id_unit, palier_group, start_date, end_date)
    VALUES
        (:fk_id_brand, :fk_id_category, :fk_id_industrial, :fk_id_unit, :palier_group, :start_date, :end_date)
""")

agreement_ids = []
for _, row in mapping_work.iterrows():
    result = connection_database.execute(insert_agreement, {
        "fk_id_brand":      int(row["id_brand"]),
        "fk_id_category":   int(row["id_category"]),
        "fk_id_industrial": int(row["id_industrial"]),
        "fk_id_unit":       id_unit_uvc,
        "palier_group":     row["palier_group"] if pd.notna(row.get("palier_group")) else None,
        "start_date":       None,
        "end_date":         None,
    })
    agreement_ids.append(result.lastrowid)

connection_database.commit()
mapping_work["id_agreement"] = agreement_ids

# Table `Agreement Tier`

> - `id_agreement_tier` (primary key, auto-increment)
> - `fk_id_agreement` → `agreement.id_agreement`
> - `min_uvc` (int, not null)
> - `max_uvc` (int, nullable)
> - `price` (decimal, not null)

In [61]:
def parse_palier_col(param_column_name):
    """ Parse a palier column name to extract the tier name, min UVC and max UVC (if applicable) """

    # Regex for palier columns with format "palier_{tier_name}_{min_uvc}-{max_uvc}_uvc"
    match = re.match(r"^palier_(.+)_(\d+)-(\d+)_uvc$", param_column_name)
    if match:
        return match.group(1), int(match.group(2)), int(match.group(3)) - 1

    # Regex for palier columns with format "palier_{tier_name}_superior-{min_uvc}_uvc"
    match = re.match(r"^palier_(.+)_superior-(\d+)_uvc$", param_column_name)
    if match:
        return match.group(1), int(match.group(2)), None

    return None

insert_tier = text("""
    INSERT INTO agreement_tier
        (fk_id_agreement, min_uvc, max_uvc, price)
    VALUES
        (:fk_id_agreement, :min_uvc, :max_uvc, :price)
""")

palier_cols = [mapping_column for mapping_column in mapping.columns if mapping_column.startswith("palier_")]
for _, row in mapping_work.iterrows():
    for palier_column in palier_cols:
        price = row[palier_column]
        if pd.isna(price):
            continue

        parsed = parse_palier_col(palier_column)
        if parsed is None:
            continue

        _, min_uvc, max_uvc = parsed
        connection_database.execute(insert_tier, {
            "fk_id_agreement": int(row["id_agreement"]),
            "min_uvc":         min_uvc,
            "max_uvc":         max_uvc,
            "price":           float(price),
        })

connection_database.commit()

# Table `Transaction`

> - `id_transaction` (primary key, auto-increment)
> - `fk_id_product` → `product.id_product`
> - `fk_id_agreement` → `agreement.id_agreement`
> - `fk_id_distributor` → `distributor.id_distributor`
> - `quantity` (int, not null)
> - `unit_price` (decimal, not null) — `amount_ht / quantity`
> - `total_price` (decimal, not null) — `amount_ht`
> - `transaction_date` (date, nullable)

In [62]:
database_product    = pd.read_sql(text("SELECT * FROM product"),    connection_database)
database_agreement  = pd.read_sql(text("SELECT * FROM agreement"),  connection_database)
database_distributor = pd.read_sql(text("SELECT * FROM distributor"), connection_database)
database_brand      = pd.read_sql(text("SELECT * FROM brand"),      connection_database)
database_category   = pd.read_sql(text("SELECT * FROM category"),   connection_database)

dataframe_transactions = pd.read_excel(
    os.path.join(root_path, "data_folder", "product_detail_export_final.xlsx")
)

mapping = pd.read_excel(
    os.path.join(root_path, "data_folder", "mapping", "mapping_product.xlsx"),
    sheet_name="mapping_products",
)

In [63]:
# Step 1: join product_name → mapping to get (brand_db, categories)
brand_name_correction = {"Hellmanns": "Hellmann's"}

mapping_ref = mapping[["product_name", "brand", "categories"]].copy()
mapping_ref["brand_db"] = mapping_ref["brand"].map(lambda b: brand_name_correction.get(b, b))

dataframe_transactions = dataframe_transactions.merge(
    mapping_ref[["product_name", "brand_db", "categories"]],
    on="product_name",
    how="left"
)

# Step 2: join brand and category tables to get their IDs
dataframe_transactions = dataframe_transactions.merge(
    database_brand.rename(columns={"brand_name": "brand_db"}),
    on="brand_db",
    how="left"
)
dataframe_transactions = dataframe_transactions.merge(
    database_category.rename(columns={"category_name": "categories"}),
    on="categories",
    how="left"
)

# Step 3: join agreement on (fk_id_brand, fk_id_category)
dataframe_transactions = dataframe_transactions.merge(
    database_agreement[["id_agreement", "fk_id_brand", "fk_id_category"]],
    left_on=["id_brand", "id_category"],
    right_on=["fk_id_brand", "fk_id_category"],
    how="left"
)

# Step 4: join product table on (product_name, product_code, description) → id_product
dataframe_transactions = dataframe_transactions.merge(
    database_product[["id_product", "product_name", "product_code", "description"]],
    on=["product_name", "product_code", "description"],
    how="left"
)

# Step 5: join distributor
dataframe_transactions = dataframe_transactions.merge(
    database_distributor.rename(columns={"distributor_name": "distributor"}),
    on="distributor",
    how="left"
)

# Step 6: compute unit_price
dataframe_transactions["unit_price"] = (
    dataframe_transactions["amount_ht"] / dataframe_transactions["quantity"]
).round(2)

In [64]:
insert_transaction = text("""
    INSERT INTO transaction
        (fk_id_product, fk_id_agreement, fk_id_distributor,
         quantity, unit_price, total_price, transaction_date)
    VALUES
        (:fk_id_product, :fk_id_agreement, :fk_id_distributor,
         :quantity, :unit_price, :total_price, :transaction_date)
""")

for _, row in dataframe_transactions.iterrows():
    connection_database.execute(insert_transaction, {
        "fk_id_product":     int(row["id_product"])    if pd.notna(row["id_product"])    else None,
        "fk_id_agreement":   int(row["id_agreement"])  if pd.notna(row["id_agreement"])  else None,
        "fk_id_distributor": int(row["id_distributor"]) if pd.notna(row["id_distributor"]) else None,
        "quantity":          int(row["quantity"]),
        "unit_price":        float(row["unit_price"]),
        "total_price":       float(row["amount_ht"]),
        "transaction_date":  None,
    })

connection_database.commit()